# Kaggle/Colab: LoRA/QLoRA-обучение ChessAnnotator за 5 минут

Ноутбук обучает LoRA-адаптер на «золотом датасете» (`schema_version=2`)
и упаковывает результат в `/kaggle/working/models/lora-club-v2-adapter.zip`.

## Что нужно

1. **Settings → Accelerator → GPU T4 ×2** (T4 16 GB; P100 не подходит —
   на нём нет bf16).
2. **Settings → Internet → On** (клонирование репозитория и загрузка
   базовой модели).
3. Загрузите `data/gold_dataset_v2.jsonl` как **Kaggle Dataset** с именем
   `chess-gold-v2` (Datasets → New Dataset → Upload).
4. Нажмите **Run All**.

> Ноутбук по умолчанию берёт **Qwen/Qwen3-4B-Instruct-2507**,
> `batch_size=1`, `grad_accum=16`, `max_length=2048`, `--no-4bit` и
> `--force-fp16`. Так обучение стабильно идёт на T4: bitsandbytes в
> Colab/Kaggle капризничает, а bf16 на T4 не поддерживается. Для v0.2.0
> ноутбук берёт `gold_dataset_v2.jsonl`, `EPOCHS = 1`, `LR = 1e-4`; 7B
> на T4 16 GB запускается только с `--load-in-4bit` и `max_length=1024`
> (ручной сценарий в GPU_TRAINING_GUIDE §11.2).

## Что получится

- `models/lora-club-v2/` — LoRA-адаптер (`adapter_config.json`, веса);
- `lora-club-v2-adapter.zip` — архив для скачивания из вкладки **Output**;
- лог лосса в `models/lora-club-v2/runs/` (TensorBoard).


## 1. Зависимости

Kaggle уже содержит `torch`, собранный под CUDA образа. Переустановка
может сломать совместимость с драйвером, поэтому команда ниже не
обновляет `torch` без необходимости: `pip` видит установленную версию и
пропускает её.

`torchao` ставим отдельно с `--no-deps`: в образе Colab/Kaggle лежит
старая версия, на которой падает `peft` (`ImportError`/`AttributeError`
при импорте). `--no-deps` не даёт pip заодно переустановить torch.


In [ ]:
# Если torch действительно отсутствует в образе, раскомментируйте:
# %pip install -q torch --index-url https://download.pytorch.org/whl/cu121
%pip install -q "torch" "transformers>=4.44" "peft>=0.13"
%pip install -q "datasets>=3.0" "accelerate>=1.0"
%pip install -q "bitsandbytes>=0.44" "tensorboard>=2.16"
%pip install -q --upgrade "torchao>=0.16.0" --no-deps


## 1.1. Блокировка TensorFlow (Colab/Kaggle)

В образе Colab встроенный TensorFlow при импорте инициализирует CUDA 13.0,
падает с `libcudart.so.12` и переключает PyTorch на CPU. Поэтому TF надо
заблокировать **до** импорта `torch`/`transformers`: переменные окружения
действуют на весь kernel, а ячейка ниже ещё и выгружает уже импортированный
`tensorflow` из `sys.modules`.

> Если ячейка выполнялась не первой (например, после `import torch`),
> выполните **Run → Restart & Run All**.


In [ ]:
import os
import sys

os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

for name in list(sys.modules):
    if name == "tensorflow" or name.startswith("tensorflow."):
        del sys.modules[name]

print(
    "TensorFlow заблокирован:",
    os.environ["USE_TF"],
    os.environ["TRANSFORMERS_NO_TF"],
)


## 2. Репозиторий

Клонируем проект с `train_lora.py` и вспомогательными модулями. Если
репозиторий приватный, добавьте GitHub-токен в Kaggle Secrets и
используйте URL вида `https://<token>@github.com/...`.


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/iilmchat/ai-chess-annotator.git"
REPO_REF = "main"
REPO_DIR = Path("/kaggle/working/ai-chess-annotator")

if REPO_DIR.exists():
    !git -C {REPO_DIR} fetch --depth 1 origin {REPO_REF}
    !git -C {REPO_DIR} reset --hard origin/{REPO_REF}
else:
    !git clone --depth 1 --branch {REPO_REF} {REPO_URL} {REPO_DIR}

os.chdir(REPO_DIR)
assert Path("scripts/train_lora.py").is_file(), "train_lora.py не найден"
commit = subprocess.check_output(
    ["git", "rev-parse", "--short", "HEAD"], text=True
).strip()
print(f"Репозиторий: {REPO_DIR} | commit: {commit}")


## 3. Проверка GPU

In [ ]:
!nvidia-smi
import torch

assert torch.cuda.is_available(), "GPU не включён: Settings → Accelerator → GPU T4"
print("torch:", torch.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))
print("bf16:", torch.cuda.is_bf16_supported())


## 4. Датасет

Ожидается `gold_dataset_v2.jsonl` из Kaggle Dataset `chess-gold-v2`. Если
датасет назван иначе, ноутбук попробует найти `gold_dataset_v2.jsonl`, `gold_dataset.jsonl` или
`lora_gold_v1.jsonl` в `/kaggle/input/*/`.


In [ ]:
DATASET_CANDIDATES = [
    Path("/kaggle/input/chess-gold-v2/gold_dataset_v2.jsonl"),
    Path("/kaggle/input/chess-gold/gold_dataset.jsonl"),
    Path("/kaggle/input/chess-gold/lora_gold_v1.jsonl"),
]
DATASET_PATH = next(
    (path for path in DATASET_CANDIDATES if path.is_file()), None
)
if DATASET_PATH is None:
    matches = sorted(Path("/kaggle/input").glob("*/gold_dataset_v2.jsonl"))
    matches += sorted(Path("/kaggle/input").glob("*/gold_dataset.jsonl"))
    matches += sorted(Path("/kaggle/input").glob("*/lora_gold_v1.jsonl"))
    DATASET_PATH = matches[0] if matches else DATASET_CANDIDATES[0]

if not DATASET_PATH.is_file():
    raise FileNotFoundError(
        f"Датасет не найден: {DATASET_PATH}. Загрузите gold_dataset_v2.jsonl "
        "как Kaggle Dataset chess-gold-v2 или задайте DATASET_PATH вручную."
    )

with DATASET_PATH.open(encoding="utf-8") as stream:
    records = sum(1 for line in stream if line.strip())
print(f"Датасет: {DATASET_PATH} | записей: {records}")


## 5. Конфигурация T4

Параметры из постановки итерации 47: `batch_size=1`, `grad_accum=16`,
`max_length=2048`, `--no-4bit`, `--force-fp16`. Режим без 4-bit требует
модели до 4B на T4 16 GB (веса 4B в fp16 — около 8 GB). 7B без
квантования в 16 GB не помещается: для неё (`Qwen/Qwen2.5-7B-Instruct`)
верните `--load-in-4bit` в команде обучения и `MAX_LENGTH = 1024`.


In [ ]:
MODEL_NAME = "Qwen/Qwen3-4B-Instruct-2507"
BATCH_SIZE = 1
GRAD_ACCUM = 16
MAX_LENGTH = 2048
EPOCHS = 1
LR = 1e-4
OUTPUT_DIR = Path("/kaggle/working/models/lora-club-v2")

# Если huggingface.co недоступен (РФ), включите зеркало до загрузки
# модели:
# os.environ["HF_ENDPOINT"] = "https://hf-mirror.com"

# 7B-модель на T4 16 GB — только с квантованием:
# MODEL_NAME = "Qwen/Qwen2.5-7B-Instruct"
# MAX_LENGTH = 1024

print(
    f"model={MODEL_NAME} batch={BATCH_SIZE} "
    f"grad_accum={GRAD_ACCUM} max_length={MAX_LENGTH} lr={LR} epochs={EPOCHS}"
)


## 6. Проверка датасета без загрузки модели

`--dry-run` проверяет `schema_version`, роли и длины записей. Если здесь
ошибка — обучение не тратит GPU.


In [ ]:
!python scripts/train_lora.py --dry-run --dataset {DATASET_PATH} --max-length {MAX_LENGTH}


## 7. Обучение

Обучение идёт с `--no-4bit` (bitsandbytes в Colab/Kaggle нестабилен) и
`--force-fp16` (T4 не поддерживает bf16, а `is_bf16_supported()` на части
сборок врёт, из-за чего обучение зависает). Лог пишется в
`OUTPUT_DIR/runs`, чекпоинты — в `OUTPUT_DIR/checkpoint-*`.


In [ ]:
cmd = [
    sys.executable,
    "scripts/train_lora.py",
    "--dataset",
    str(DATASET_PATH),
    "--model",
    MODEL_NAME,
    "--output-dir",
    str(OUTPUT_DIR),
    "--epochs",
    str(EPOCHS),
    "--lr",
    str(LR),
    "--batch-size",
    str(BATCH_SIZE),
    "--grad-accum",
    str(GRAD_ACCUM),
    "--max-length",
    str(MAX_LENGTH),
    "--no-4bit",
    "--force-fp16",
    "--gradient-checkpointing",
    "--report_to",
    "tensorboard",
    "--seed",
    "42",
]
print("Запуск:", " ".join(cmd))
subprocess.run(cmd, check=True)


## 8. TensorBoard (опционально)

В Kaggle расширение TensorBoard доступно не всегда. Если оно есть,
раскомментируйте команды в ячейке ниже; иначе скачайте `runs/` и
откройте локально.


In [ ]:
# %load_ext tensorboard
# %tensorboard --logdir /kaggle/working/models/lora-club-v2/runs


## 9. Упаковка адаптера

Архив появится в `/kaggle/working` и станет доступен во вкладке
**Output** после завершения сессии.


In [ ]:
import shutil

adapter_config = OUTPUT_DIR / "adapter_config.json"
assert adapter_config.is_file(), (
    f"Нет {adapter_config}: обучение не завершилось или output_dir другой"
)
archive = shutil.make_archive(
    base_name=f"{OUTPUT_DIR}-adapter",
    format="zip",
    root_dir=str(OUTPUT_DIR.parent),
    base_dir=OUTPUT_DIR.name,
)
print("Архив:", archive)


## 10. Продолжение после обрыва сессии

Kaggle-сессия ограничена (~12 ч, на практике меньше). Если в
`OUTPUT_DIR` остались `checkpoint-*`, продолжите обучение с последнего
чекпоинта, а не начинайте заново. Если сессия уничтожила каталог,
загрузите прошлый zip как Dataset и распакуйте его в `OUTPUT_DIR`.


In [ ]:
# RESUME = Path("/kaggle/working/models/lora-club-v2/checkpoint-500")
# resume_cmd = cmd + ["--resume_from_checkpoint", str(RESUME)]
# subprocess.run(resume_cmd, check=True)


## Траблшутинг

| Симптом | Что делать |
| :--- | :--- |
| `libcudart.so.12` / TensorFlow инициализирует CUDA | выполнить ячейку **1.1** до импорта torch; затем **Restart & Run All** |
| `ImportError: torchao` / падение `peft` | `%pip install -q --upgrade "torchao>=0.16.0" --no-deps`, затем **Restart & Run All** |
| Обучение зависает на T4 | T4 не поддерживает bf16: запускать с `--force-fp16` (в ноутбуке уже есть) |
| `CUDA out of memory` (OOM) | уменьшить `MAX_LENGTH` до `1024`; для 7B вернуть `--load-in-4bit`; проверить, что выбрана T4, а не P100 |
| `Датасет не найден` | проверить имя Kaggle Dataset (`chess-gold-v2`) и файл `gold_dataset_v2.jsonl`; задать `DATASET_PATH` вручную |
| Скачивание Qwen зависает | включить Internet; при блокировке HF задать `HF_ENDPOINT=https://hf-mirror.com` перед ячейкой обучения |
| `ModuleNotFoundError` после `%pip` | выполнить **Run → Restart & Run All** (ядро подхватит новые пакеты) |
| Сессия оборвалась | взять `checkpoint-*` из Output, распаковать в `OUTPUT_DIR` и использовать `--resume_from_checkpoint` |

После обучения скачайте `lora-club-v2-adapter.zip`, распакуйте на
основной машине и укажите `LORA_ADAPTER_PATH=models/lora-club-v2`
(см. `docs/development/GPU_TRAINING_GUIDE.md` §7–§8).
